# NB12 – Alternative Speichertechnologien
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Nicht-elektrochemische Grossspeicher als Systemergänzung zum Batterie-Business-Case:  
**Pumpspeicher · CAES · Power-to-X · Schwungradspeicher**

---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [← NB11 Technologievergleich](11_Technologievergleich.ipynb) | [→ NB13 Eigenverbrauch](13_Eigenverbrauch.ipynb) |
|:---|:---:|---:|


---
## 0. Setup

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

with open('config.json') as f:
    CFG = json.load(f)

SZ_AKTIV   = CFG.get('gleichzeitigkeit_aktiv', 'realistisch')
CHARTS_DIR = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
_viz=CFG.get('visualisierung',{}).get('farben',{})
BG_DARK  = _viz.get('bg_dark',  '#0d1117')
BG_PANEL = _viz.get('bg_panel', '#141414')
C_PRICE  = _viz.get('c_price',  '#FFA726')
C_LOAD   = _viz.get('c_load',   '#66BB6A')
C_CHARGE = _viz.get('c_charge', '#1565C0')
C_FEED   = _viz.get('c_feed',   '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])
print(f"Setup OK | Charts → {CHARTS_DIR}")


---
## 1. Motivation: Warum alternative Speicher?

Batterien dominieren den Kleinst- und Mittelsegmentmarkt (< 10 MWh).
Für grosse Energiemengen (> 100 MWh) und lange Entladezeiten (> 8 h) sind
mechanische und chemische Speicher oft günstiger und dauerhafter.

In der Schweiz ist **Pumpspeicher** bereits etabliert — er ist mit Abstand
der grösste Energiespeicher des Landes und unverzichtbar für die Stabilität
des europäischen Verbundnetzes.


---
## 2. Steckbriefe

### 2.1 Pumpspeicher (Pumped Hydro Storage — PHS)

| Kennwert | Wert |
|---|---|
| **Typische Kapazität** | 100 MWh … 10 GWh |
| **System-CAPEX** | 5–15 EUR/kWh (bei grosser Kapazität) |
| **Lebensdauer** | 50–100 Jahre |
| **Round-Trip-Wirkungsgrad** | 70–85 % |
| **Selbstentladung** | praktisch 0 |
| **Reaktionszeit** | 1–5 Minuten (Anlauf) |
| **CH-Relevanz** | ★★★★★ — wichtigster CH-Speicher |

Überschussstrom pumpt Wasser in ein Oberbecken. Bei Bedarf fliesst es durch
Turbinen zurück. Die Schweiz betreibt ~9.5 GW installierte Pumpspeicherleistung
(weltweit Spitzenposition pro Kopf).

*CH-Beispiele: Nant de Drance (900 MW, VS) · Linth-Limmern (1 GW, GL) · Grimsel KWO (700 MW, BE)*

**Grenzen:** Geographisch gebunden (Topographie), lange Planungs-/Bauzeit,
politische Hürden. Keine dezentrale Verteilung möglich.

---

### 2.2 Druckluftspeicher (CAES — Compressed Air Energy Storage)

| Kennwert | Wert |
|---|---|
| **Typische Kapazität** | 100 MWh … 5 GWh |
| **System-CAPEX** | 50–100 EUR/kWh |
| **Lebensdauer** | 30–40 Jahre |
| **Round-Trip-Wirkungsgrad** | 40–70 % (A-CAES: bis 70 %) |
| **Reaktionszeit** | Minuten |
| **CH-Relevanz** | ★★ — Geologie ungeeignet für klassisches CAES |

Überschussstrom komprimiert Luft in Kavernen oder Druckbehälter; zur
Stromerzeugung expandiert sie durch Turbinen.

- **A-CAES (adiabat):** Kompressionswärme wird gespeichert und zurückgeführt → RTE bis 70 %, kein Gas nötig
- **D-CAES (diabat):** Benötigt Gas zur Nacherhitzung → RTE ~55 %

**Grenzen in CH:** Keine natürlichen Salzkavernen (wie in D/US); Felskavernen teuer.
Eher für Norddeutschland/Nordsee-Raum relevant.

---

### 2.3 Power-to-X (P2X)

| Kennwert | Wert |
|---|---|
| **Varianten** | Power-to-H₂, Power-to-CH₄, Power-to-Heat, Power-to-Liquid |
| **CAPEX Elektrolyseur** | 500–1 000 EUR/kW_el |
| **Wirkungsgrad Elektrolyse** | 60–80 % |
| **Gesamt-RTE (Strom→Strom)** | 25–45 % |
| **Speicherdauer** | Tage … saisonal |
| **CH-Relevanz** | ★★★ — Pilotprojekte aktiv, saisonal interessant |

Überschussstrom (z.B. Sommer-Solar) wird zu H₂ oder synthetischem Methan
und im Gasnetz gespeichert. Für saisonalen Ausgleich (Solar-Sommer → Winter-Heizung)
strategisch interessant; für tägliche Arbitrage wegen sehr niedrigem RTE ungeeignet.

*CH-Projekte: Hybridwerk Aarmatt (Repower, BE) · H2 Volketswil (ZH) · HyFly (Region Zürich-Flughafen)*

---

### 2.4 Schwungradspeicher (Flywheel Energy Storage — FES)

| Kennwert | Wert |
|---|---|
| **Kapazität Einzeleinheit** | 25–125 kWh (moderne Komposit-Rotoren) |
| **Kapazität Array/Farm** | 1–80 MWh (modular kombiniert) |
| **CAPEX** | 1 000–5 000 EUR/kWh (Einzeleinheit) |
| **Lebensdauer** | > 20 Jahre, > 1 000 000 Zyklen |
| **Round-Trip-Wirkungsgrad** | 90–95 % (Magnetlagerung, Vakuum) |
| **Selbstentladung** | 2–20 %/h (stark systemabhängig) |
| **CH-Relevanz** | ★★ — Grid-FCR, USV, Militär-Mikronetze |

Einzeleinheiten haben hohe Selbstentladung (≥5 %/h mit mechanischen Lagern, <1 %/Tag
bei Supraleitungs-Magnetlagerung) — tägliche Arbitrage unwirtschaftlich.
Stärken: Millisekunden-Reaktionszeit, praktisch unbegrenzte Zyklenanzahl, temperaturunabhängig.

**Grossskalige Anlagen (Beispiele):**
- Beacon Power, Stephentown NY: 20 MW / 5 MWh (200 Einheiten, Frequenzregelung)
- Amber Kinetics / PG&E, Fresno CA: 20 MW / 80 MWh (4h Entladezeit)
- China: 30 MW Gridanlage, in Betrieb seit 2024

**Militärische Anwendungen:**
Schwungräder eignen sich für Puls-Energiespeicherung bei Hochenergie-Systemen
(z.B. elektromagnetische Startanlagen EMALS auf Flugzeugträgern, Railgun-Energieversorgung).
Vorteile: keine Degradation, keine Temperaturempfindlichkeit, kein Brandrisiko —
wichtig in geschlossenen militärischen Plattformen (Schiffe, U-Boote, Fahrzeuge).


---
## 3. Positionierung: Entladezeit vs. Wirkungsgrad


In [ ]:
# ── Bubble-Chart: Speichertechnologien ───────────────────────────────────────
# x = typische Entladezeit [h], y = RTE [%], Bubble-Grösse ~ CAPEX EUR/kWh
names    = ['Li-Ion (LFP)', 'Redox-Flow',  'Pumpspeicher', 'CAES (adiabat)', 'Power-to-X', 'Schwungrad']
dh_typ   = [4,             8,             10,              6,                100,           0.25]
rte      = [93,            75,            80,              65,                35,            92]
capex_n  = [225,           600,           10,              75,               400,           2500]  # EUR/kWh
colors_b = ['#42A5F5','#66BB6A','#26C6DA','#FFA726','#AB47BC','#EF5350']

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors='#bbbbbb')
for sp in ax.spines.values(): sp.set_edgecolor('#333')

for name, dh, r, cx, col in zip(names, dh_typ, rte, capex_n, colors_b):
    size = max(80, min(2800, cx * 2.2))
    ax.scatter(dh, r, s=size, color=col, alpha=0.75, edgecolors='white', lw=0.8)
    ax.annotate(f'{name}\n({cx} \u20ac/kWh)', (dh, r),
                textcoords='offset points', xytext=(10, 4),
                color='white', fontsize=8.5)

ax.axhline(80, color='#555', lw=0.8, ls=':', alpha=0.7)
ax.text(0.06, 81, 'RTE-Schwelle 80 %', color='#666', fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Typische Entladezeit [h]  (log-Skala)', color='#aaa', fontsize=11)
ax.set_ylabel('Round-Trip-Wirkungsgrad [%]', color='#aaa', fontsize=11)
ax.set_title('Speichertechnologien — Entladezeit vs. Wirkungsgrad\n(Bubble-Grösse \u221d System-CAPEX [EUR/kWh])',
             color='white', fontweight='bold')
ax.grid(True, alpha=0.12)
ax.set_xlim(0.02, 300)

p = os.path.join(CHARTS_DIR, 'kuer_nb12_positionierung.png')
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")


**Lesehilfe** (log-Skala X-Achse):
- **Oben Mitte/Rechts** — Langer Speicher + hoher Wirkungsgrad: Li-Ion (4h Utility-Standard) und Pumpspeicher — beide kommerziell etabliert
- **Oben Links** — Sehr kurze Entladezeit + hoher Wirkungsgrad: Schwungrad-Farmen (FCR, 15 min) — Nische Frequenzregelung
- **Mitte** — Mittlere Entladezeit, mittlerer Wirkungsgrad: Redox-Flow (8h) und CAES (6h) — Utility-Grossspeicher
- **Rechts Unten** — Sehr langer Speicher, niedriger Wirkungsgrad: Power-to-X (Tage–saisonal) — Sektorkopplung, nicht für Arbitrage

> **Hinweis Li-Ion:** Der Datenpunkt zeigt 4h (NREL ATB Utility-Standard). Li-Ion deckt tatsächlich 1–10h ab — von Residential (1–2h) bis Utility Long-Duration (8–10h). Die Entladezeit ist konfigurierbar, nicht technologiebedingt fix.

> **Hinweis Schwungrad:** Einzeleinheiten laden in Sekunden, Farmen erreichen 15–60 min effektive Entladezeit. Kapazitäten bis 80 MWh möglich (Arrays).


---
## 4. Einordnung im CH-Kontext

| Technologie | Rol im CH-Energiesystem | Arbitrage-Eignung |
|---|---|---|
| **Pumpspeicher** | Hauptspeicher für saisonalen Ausgleich, Regelenergie | Gut (Grossskala) |
| **Li-Ion (LFP)** | Dezentral, schnell reagierend, modular | Sehr gut (alle Segmente) |
| **Redox-Flow** | Utility-Grossspeicher, langer Zyklushorizont | Gut (> 10 MWh) |
| **CAES** | Derzeit nicht relevant in CH (Geologie) | Bedingt |
| **Power-to-X** | Saisonaler Speicher, Sektorkopplung Strom/Gas/Wärme | Nicht für Arbitrage |
| **Schwungrad** | Netzregelung (FCR), USV — kein Energiespeicher im eigentlichen Sinn | Nein |

**Fazit für das Projekt:**  
Grid-Arbitrage mit Batterien und Pumpspeichern sind **komplementär** —
nicht kompetitiv. Batterien agieren im Tages- und Wochenspeicher-Bereich,
Pumpspeicher sichern den saisonalen Ausgleich und die Frequenzstabilität.
Power-to-X schliesst die Lücke für mehrmonatige Speicherung.


---

In [ ]:
# ── Abschlusskontrolle ───────────────────────────────────────────────────────
charts_out = [f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_nb12_')]
print("=== Abschlusskontrolle NB12 (Kür) ===")
for f in sorted(charts_out):
    print(f"  ✅  {f}")
print(f"  {len(charts_out)} Chart(s) erzeugt")
print("=== Ende ===")


---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [← NB11 Technologievergleich](11_Technologievergleich.ipynb) | [→ NB13 Eigenverbrauch](13_Eigenverbrauch.ipynb) |
|:---|:---:|---:|
